In [ ]:
"""
Task:
    Construct focal new hires in O*NET major groups 17 and 19.

Inputs:
(a) user_positions (Microsoft Fabric table)
(b) Embedded O*NET code-title mapping derived from the project reference CSV

Outputs:
(a) Files/WenzhiW/B01_ConstructAnalysisSample/FocalNewHires_AllIndustries

Description of outputs:
(1) Data (a) is a multi-part Parquet dataset with one focal spell per user-company pair.

Notes:
(1) Raw occupation and start-date filters precede general cleaning.
(2) The sample covers all industries and starts in 2021-2023.
(3) Internal transfers are retained and rehires are not separately identified.

Time: 2026-08-19
"""

import re

from pyspark import StorageLevel
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, IntegralType, StringType, TimestampType


# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 0. Specify parameters, the O*NET universe, and mechanical helpers
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


POSITION_TABLE = "user_positions"
OUTPUT_PARQUET = "Files/WenzhiW/B01_ConstructAnalysisSample/FocalNewHires_AllIndustries"
OUTPUT_WRITE_MODE = "overwrite"

COHORT_START_DATE = "2021-01-01"
COHORT_END_DATE_EXCLUSIVE = "2024-01-01"

SOURCE_COLUMNS = (
    "position_id",
    "user_id",
    "rcid",
    "position_number",
    "startdate",
    "country",
    "state",
    "title_raw",
    "title_translated",
    "seniority",
    "rics_k400",
    "naics_code",
    "naics_description",
    "onet_code",
    "onet_title",
)

MISSING_TEXT_VALUES = (
    "",
    "empty",
    "null",
    "none",
    "nan",
    "na",
    "n/a",
)
TEXT_PUNCTUATION_REGEX = r"[\p{P}\p{S}]+"

INTERNSHIP_TITLE_REGEX = (
    r"(?i)(?:\bintern(ship)?s?\b|\bsummer[\s-]+interns?\b|"
    r"\bsummer[\s-]+students?\b|\bstudent[\s-]+interns?\b|"
    r"\bworking[\s-]+students?\b|\bplacement[\s-]+students?\b|"
    r"\bwork[\s-]+placements?\b|\bindustrial[\s-]+placements?\b|"
    r"\bco[\s-]?ops?\b|\bco[\s-]?operative students?\b)"
)

# Source: data/a_raw_data/B_ONET/All_Occupations.csv.
# Embedded on 2026-08-19: 125 codes from major groups 17 and 19.
# The tuple retains duplicate entries long enough for the assertions below to detect them.
FOCAL_ONET_SNAPSHOT_VERSION = "All_Occupations.csv@2026-08-19"
FOCAL_ONET_ITEMS = (
    ("17-3021.00", "Aerospace Engineering and Operations Technologists and Technicians"),
    ("17-2011.00", "Aerospace Engineers"),
    ("17-2021.00", "Agricultural Engineers"),
    ("19-4012.00", "Agricultural Technicians"),
    ("19-1011.00", "Animal Scientists"),
    ("19-3091.00", "Anthropologists and Archeologists"),
    ("17-1011.00", "Architects, Except Landscape and Naval"),
    ("17-3011.00", "Architectural and Civil Drafters"),
    ("19-2011.00", "Astronomers"),
    ("19-2021.00", "Atmospheric and Space Scientists"),
    ("17-3027.01", "Automotive Engineering Technicians"),
    ("17-2141.02", "Automotive Engineers"),
    ("19-1021.00", "Biochemists and Biophysicists"),
    ("17-2031.00", "Bioengineers and Biomedical Engineers"),
    ("19-1029.01", "Bioinformatics Scientists"),
    ("19-1029.00", "Biological Scientists, All Other"),
    ("19-4021.00", "Biological Technicians"),
    ("19-1029.04", "Biologists"),
    ("17-3028.00", "Calibration Technologists and Technicians"),
    ("17-1021.00", "Cartographers and Photogrammetrists"),
    ("17-2041.00", "Chemical Engineers"),
    ("19-4031.00", "Chemical Technicians"),
    ("19-2031.00", "Chemists"),
    ("17-3022.00", "Civil Engineering Technologists and Technicians"),
    ("17-2051.00", "Civil Engineers"),
    ("19-2041.01", "Climate Change Policy Analysts"),
    ("19-3033.00", "Clinical and Counseling Psychologists"),
    ("19-3039.03", "Clinical Neuropsychologists"),
    ("17-2061.00", "Computer Hardware Engineers"),
    ("19-1031.00", "Conservation Scientists"),
    ("17-3019.00", "Drafters, All Other"),
    ("19-3011.00", "Economists"),
    ("17-3023.00", "Electrical and Electronic Engineering Technologists and Technicians"),
    ("17-3012.00", "Electrical and Electronics Drafters"),
    ("17-2071.00", "Electrical Engineers"),
    ("17-3024.00", "Electro-Mechanical and Mechatronics Technologists and Technicians"),
    ("17-2072.00", "Electronics Engineers, Except Computer"),
    ("17-2199.03", "Energy Engineers, Except Wind and Solar"),
    ("17-3029.00", "Engineering Technologists and Technicians, Except Drafters, All Other"),
    ("17-2199.00", "Engineers, All Other"),
    ("19-3011.01", "Environmental Economists"),
    ("17-3025.00", "Environmental Engineering Technologists and Technicians"),
    ("17-2081.00", "Environmental Engineers"),
    ("19-2041.02", "Environmental Restoration Planners"),
    ("19-4042.00", "Environmental Science and Protection Technicians, Including Health"),
    ("19-2041.00", "Environmental Scientists and Specialists, Including Health"),
    ("19-1041.00", "Epidemiologists"),
    ("17-2111.02", "Fire-Prevention and Protection Engineers"),
    ("19-4013.00", "Food Science Technicians"),
    ("19-1012.00", "Food Scientists and Technologists"),
    ("19-4092.00", "Forensic Science Technicians"),
    ("19-4071.00", "Forest and Conservation Technicians"),
    ("19-1032.00", "Foresters"),
    ("17-2141.01", "Fuel Cell Engineers"),
    ("19-1029.03", "Geneticists"),
    ("17-1022.01", "Geodetic Surveyors"),
    ("19-3092.00", "Geographers"),
    ("19-4043.00", "Geological Technicians, Except Hydrologic Technicians"),
    ("19-2042.00", "Geoscientists, Except Hydrologists and Geographers"),
    ("17-2111.00", "Health and Safety Engineers, Except Mining Safety Engineers and Inspectors"),
    ("19-3093.00", "Historians"),
    ("17-2112.01", "Human Factors Engineers and Ergonomists"),
    ("19-4044.00", "Hydrologic Technicians"),
    ("19-2043.00", "Hydrologists"),
    ("19-2041.03", "Industrial Ecologists"),
    ("17-3026.00", "Industrial Engineering Technologists and Technicians"),
    ("17-2112.00", "Industrial Engineers"),
    ("19-3032.00", "Industrial-Organizational Psychologists"),
    ("17-1012.00", "Landscape Architects"),
    ("19-1099.00", "Life Scientists, All Other"),
    ("19-4099.00", "Life, Physical, and Social Science Technicians, All Other"),
    ("17-2112.03", "Manufacturing Engineers"),
    ("17-2121.00", "Marine Engineers and Naval Architects"),
    ("17-2131.00", "Materials Engineers"),
    ("19-2032.00", "Materials Scientists"),
    ("17-3013.00", "Mechanical Drafters"),
    ("17-3027.00", "Mechanical Engineering Technologists and Technicians"),
    ("17-2141.00", "Mechanical Engineers"),
    ("17-2199.05", "Mechatronics Engineers"),
    ("19-1042.00", "Medical Scientists, Except Epidemiologists"),
    ("19-1022.00", "Microbiologists"),
    ("17-2199.06", "Microsystems Engineers"),
    ("17-2151.00", "Mining and Geological Engineers, Including Mining Safety Engineers"),
    ("19-1029.02", "Molecular and Cellular Biologists"),
    ("17-2199.09", "Nanosystems Engineers"),
    ("17-3026.01", "Nanotechnology Engineering Technologists and Technicians"),
    ("19-3039.02", "Neuropsychologists"),
    ("17-3029.01", "Non-Destructive Testing Specialists"),
    ("17-2161.00", "Nuclear Engineers"),
    ("19-4051.02", "Nuclear Monitoring Technicians"),
    ("19-4051.00", "Nuclear Technicians"),
    ("19-5011.00", "Occupational Health and Safety Specialists"),
    ("19-5012.00", "Occupational Health and Safety Technicians"),
    ("19-1031.03", "Park Naturalists"),
    ("17-2171.00", "Petroleum Engineers"),
    ("17-2199.07", "Photonics Engineers"),
    ("17-3029.08", "Photonics Technicians"),
    ("19-2099.00", "Physical Scientists, All Other"),
    ("19-2012.00", "Physicists"),
    ("19-3094.00", "Political Scientists"),
    ("19-4012.01", "Precision Agriculture Technicians"),
    ("19-3039.00", "Psychologists, All Other"),
    ("19-4099.01", "Quality Control Analysts"),
    ("17-2072.01", "Radio Frequency Identification Device Specialists"),
    ("19-1031.02", "Range Managers"),
    ("19-2099.01", "Remote Sensing Scientists and Technologists"),
    ("19-4099.03", "Remote Sensing Technicians"),
    ("17-2199.08", "Robotics Engineers"),
    ("17-3024.01", "Robotics Technicians"),
    ("19-3034.00", "School Psychologists"),
    ("19-4061.00", "Social Science Research Assistants"),
    ("19-3099.00", "Social Scientists and Related Workers, All Other"),
    ("19-3041.00", "Sociologists"),
    ("19-1013.00", "Soil and Plant Scientists"),
    ("17-2199.11", "Solar Energy Systems Engineers"),
    ("19-3022.00", "Survey Researchers"),
    ("17-3031.00", "Surveying and Mapping Technicians"),
    ("17-1022.00", "Surveyors"),
    ("17-2051.01", "Transportation Engineers"),
    ("19-3099.01", "Transportation Planners"),
    ("19-3051.00", "Urban and Regional Planners"),
    ("17-2112.02", "Validation Engineers"),
    ("17-2051.02", "Water/Wastewater Engineers"),
    ("17-2199.10", "Wind Energy Engineers"),
    ("19-1023.00", "Zoologists and Wildlife Biologists"),
)

FOCAL_ONET_CODES = tuple(code for code, _ in FOCAL_ONET_ITEMS)
FOCAL_ONET_TITLES = dict(FOCAL_ONET_ITEMS)

assert len(FOCAL_ONET_CODES) == len(set(FOCAL_ONET_CODES)), "O*NET codes must be unique."
assert all(re.fullmatch(r"(?:17|19)-\d{4}\.\d{2}", code) for code in FOCAL_ONET_CODES)
assert all(code.startswith(("17-", "19-")) for code in FOCAL_ONET_CODES)
assert all(title.strip() for _, title in FOCAL_ONET_ITEMS)
assert sum(code.startswith("17-") for code in FOCAL_ONET_CODES) == 59
assert sum(code.startswith("19-") for code in FOCAL_ONET_CODES) == 66
assert len(FOCAL_ONET_CODES) == 125


def as_column(column_or_name):
    """Return a Spark Column when given either a column name or a Column."""

    if isinstance(column_or_name, str):
        return F.col(column_or_name)
    return column_or_name


def clean_text(column_or_name):
    """Trim text and convert delivered missing-value strings to Spark null."""

    text = F.trim(as_column(column_or_name).cast("string"))
    is_missing = text.isNull() | F.lower(text).isin(*MISSING_TEXT_VALUES)
    return F.when(is_missing, F.lit(None).cast("string")).otherwise(text)


def normalize_title(column_or_name):
    """Normalize a reported title for the transparent internship restriction."""

    cleaned = clean_text(column_or_name)
    normalized = F.lower(cleaned)
    normalized = F.regexp_replace(normalized, r"&", " and ")
    normalized = F.regexp_replace(normalized, TEXT_PUNCTUATION_REGEX, " ")
    normalized = F.trim(F.regexp_replace(normalized, r"\s+", " "))
    unusable = cleaned.isNull() | (normalized == "")
    return F.when(unusable, F.lit(None).cast("string")).otherwise(normalized)


def validate_required_columns(data, required_columns, dataset_name):
    """Fail from schema metadata when a required source column is absent."""

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"{dataset_name} is missing required columns: {missing_columns}.")


def clean_integral(column_name, source_type):
    """Preserve native integers and safely parse text without using floating point."""

    if isinstance(source_type, IntegralType):
        return F.col(column_name)
    if isinstance(source_type, StringType):
        return F.expr(f"try_cast(trim(`{column_name}`) as bigint)")
    raise TypeError(
        f"{column_name} must be an integral or text field; found {source_type.simpleString()}."
    )


def count_true(condition, alias):
    """Count rows meeting an invariant-failure condition, returning zero for no rows."""

    return F.coalesce(F.sum(condition.cast("long")), F.lit(0).cast("long")).alias(alias)


spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

reference_onet_map = F.create_map(
    *[
        literal
        for code, title in FOCAL_ONET_ITEMS
        for literal in (F.lit(code), F.lit(title))
    ]
)

## Step 1. Restrict the raw table before general cleaning

This step checks schema metadata, projects only the 15 contracted source fields, and applies the
exact raw O*NET and half-open date filters. Defining this lazy query does not scan the positions
table.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 1. Restrict the raw table to candidate focal new hires
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>
# >> S-1-1. Validate source metadata and inspect the start-date type
# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>


user_positions_source = spark.read.table(POSITION_TABLE)
validate_required_columns(user_positions_source, SOURCE_COLUMNS, POSITION_TABLE)

source_types = {
    field.name: field.dataType
    for field in user_positions_source.schema.fields
    if field.name in SOURCE_COLUMNS
}
startdate_type = source_types["startdate"]
startdate_sql_type = startdate_type.simpleString()

if isinstance(startdate_type, DateType):
    startdate_kind = "date"
elif isinstance(startdate_type, TimestampType) or startdate_sql_type.startswith("timestamp"):
    startdate_kind = "timestamp"
elif isinstance(startdate_type, StringType):
    startdate_kind = "text"
else:
    raise TypeError(
        "startdate must be a date, timestamp, or text field; "
        f"found {startdate_sql_type}."
    )

print(f"Validated {POSITION_TABLE} schema; startdate type is {startdate_sql_type}.")


# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>
# >> S-1-2. Define the narrow raw-candidate query
# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>


"""
Notes:
(1) select projects only the contracted source fields before record-level transformations.
(2) isin applies exact delivered O*NET codes from the embedded 125-code universe.
(3) Date and timestamp fields are filtered directly; text receives one safe date parse.
(4) No title, identifier, geography, industry, or occupation-label cleaning occurs here.
"""
projected_positions = user_positions_source.select(*SOURCE_COLUMNS)

if startdate_kind == "text":
    raw_candidate_focal_new_hires = (
        projected_positions.filter(F.col("onet_code").isin(*FOCAL_ONET_CODES))
        .withColumn(
            "_initial_start_date",
            F.expr("try_cast(`startdate` as date)"),
        )
        .filter(
            (F.col("_initial_start_date") >= F.lit(COHORT_START_DATE).cast("date"))
            & (
                F.col("_initial_start_date")
                < F.lit(COHORT_END_DATE_EXCLUSIVE).cast("date")
            )
        )
    )
else:
    raw_candidate_focal_new_hires = (
        projected_positions.filter(F.col("onet_code").isin(*FOCAL_ONET_CODES))
        .filter(
            (
                F.col("startdate")
                >= F.lit(COHORT_START_DATE).cast(startdate_sql_type)
            )
            & (
                F.col("startdate")
                < F.lit(COHORT_END_DATE_EXCLUSIVE).cast(startdate_sql_type)
            )
        )
        .withColumn("_initial_start_date", F.col("startdate").cast("date"))
    )

## Step 2. Clean only the restricted candidates

All cleaning below is downstream of the selective occupation and date query. Built-in Spark
expressions keep record-level work distributed, and both reported title fields remain available
for auditing.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 2. Clean only the restricted candidate records
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


"""
Notes:
(1) Native integral identifiers retain their delivered types; text identifiers use try_cast.
(2) The minimally parsed date from Step 1 is reused to construct month and cohort year.
(3) Text cleaning converts trimmed missing-value sentinels to null.
(4) Title normalization removes only punctuation, symbols, and repeated whitespace.
"""
candidate_clean_base = raw_candidate_focal_new_hires.select(
    clean_integral("position_id", source_types["position_id"]).alias("position_id"),
    clean_integral("user_id", source_types["user_id"]).alias("user_id"),
    clean_integral("rcid", source_types["rcid"]).alias("rcid"),
    clean_integral("position_number", source_types["position_number"]).alias(
        "position_number"
    ),
    F.trunc("_initial_start_date", "month").alias("start_month"),
    F.year("_initial_start_date").cast("int").alias("start_year"),
    clean_text("country").alias("country"),
    clean_text("state").alias("state"),
    clean_text("title_raw").alias("title_raw"),
    clean_text("title_translated").alias("title_translated"),
    clean_integral("seniority", source_types["seniority"]).alias("seniority"),
    clean_text("rics_k400").alias("rics_k400"),
    clean_text("naics_code").alias("naics_code"),
    clean_text("naics_description").alias("naics_description"),
    clean_text("onet_code").alias("onet_code"),
    clean_text("onet_title").alias("onet_title"),
)

candidate_titles_ready = candidate_clean_base.select(
    "*",
    normalize_title("title_raw").alias("_title_raw_normalized"),
    normalize_title("title_translated").alias("_title_translated_normalized"),
)

candidate_clean = candidate_titles_ready.select(
    *[
        name
        for name in candidate_clean_base.columns
    ],
    F.coalesce(
        F.col("_title_translated_normalized"),
        F.col("_title_raw_normalized"),
    ).alias("restriction_title"),
    F.when(
        F.col("_title_translated_normalized").isNotNull(),
        F.lit("title_translated"),
    )
    .when(
        F.col("_title_raw_normalized").isNotNull(),
        F.lit("title_raw"),
    )
    .otherwise(F.lit(None).cast("string"))
    .alias("restriction_title_source"),
    F.element_at(reference_onet_map, F.col("onet_code")).alias(
        "reference_onet_title"
    ),
)

## Step 3. Construct flags and report sequential attrition

One projection creates every individual and cumulative restriction flag. The compact stage
expansion then supplies all six attrition rows through one grouped aggregation and one small
driver collection.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 3. Apply sample restrictions and construct attrition
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>
# >> S-3-1. Construct all individual and cumulative flags in one projection
# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>


pass_required_identifiers = (
    F.col("position_id").isNotNull()
    & F.col("user_id").isNotNull()
    & F.col("rcid").isNotNull()
    & F.col("position_number").isNotNull()
)
pass_valid_start_month = (
    F.col("start_month").isNotNull()
    & (F.col("start_month") >= F.lit(COHORT_START_DATE).cast("date"))
    & (F.col("start_month") < F.lit(COHORT_END_DATE_EXCLUSIVE).cast("date"))
)
pass_complete_geography = F.col("country").isNotNull() & F.col("state").isNotNull()
pass_usable_title = F.col("restriction_title").isNotNull()
internship_title_flag = F.coalesce(
    F.col("restriction_title").rlike(INTERNSHIP_TITLE_REGEX),
    F.lit(False),
)
pass_non_internship = ~internship_title_flag

retained_after_identifiers = pass_required_identifiers
retained_after_start_month = retained_after_identifiers & pass_valid_start_month
retained_after_geography = retained_after_start_month & pass_complete_geography
retained_after_title = retained_after_geography & pass_usable_title
retained_after_internship = retained_after_title & pass_non_internship

STEP4_CANDIDATE_COLUMNS = (
    "position_id",
    "user_id",
    "rcid",
    "position_number",
    "start_month",
    "start_year",
    "country",
    "state",
    "title_raw",
    "title_translated",
    "restriction_title",
    "restriction_title_source",
    "seniority",
    "rics_k400",
    "naics_code",
    "naics_description",
    "onet_code",
    "onet_title",
    "reference_onet_title",
)

"""
Notes:
(1) Each pass flag evaluates one rule without deleting a candidate record.
(2) Cumulative flags reproduce the specified sequential restriction order.
(3) The internship regex is applied only to the translated-first fallback title.
(4) The projection retains only fields needed by attrition or one-spell selection.
"""
candidate_flagged = candidate_clean.select(
    *STEP4_CANDIDATE_COLUMNS,
    pass_required_identifiers.cast("int").alias("pass_required_identifiers"),
    pass_valid_start_month.cast("int").alias("pass_valid_start_month"),
    pass_complete_geography.cast("int").alias("pass_complete_geography"),
    pass_usable_title.cast("int").alias("pass_usable_title"),
    internship_title_flag.cast("int").alias("internship_title_flag"),
    pass_non_internship.cast("int").alias("pass_non_internship"),
    F.lit(1).cast("int").alias("retained_raw_candidate"),
    retained_after_identifiers.cast("int").alias("retained_after_identifiers"),
    retained_after_start_month.cast("int").alias("retained_after_start_month"),
    retained_after_geography.cast("int").alias("retained_after_geography"),
    retained_after_title.cast("int").alias("retained_after_title"),
    retained_after_internship.cast("int").alias("retained_after_internship"),
)


# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>
# >> S-3-2. Inspect the plan and materialize the reused candidate projection lazily
# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>


"""
Notes:
(1) explain prints the plan without scanning the full table.
(2) Confirm projection and occupation/date filters in the plan shown by Fabric.
(3) Claim source-level pushdown only if the physical plan explicitly reports it.
(4) MEMORY_AND_DISK is justified because attrition and Step 4 reuse this narrow data.
"""
candidate_flagged.explain("formatted")
print("Review the formatted plan above for projection, filters, and reported pushdown.")
candidate_flagged = candidate_flagged.persist(StorageLevel.MEMORY_AND_DISK)


# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>
# >> S-3-3. Calculate six-stage attrition with one grouped aggregation
# >>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>#>>


ATTRITION_STAGES = (
    (0, "RAW_CANDIDATES", "retained_raw_candidate"),
    (1, "AFTER_REQUIRED_IDENTIFIERS", "retained_after_identifiers"),
    (2, "AFTER_VALID_START_MONTH", "retained_after_start_month"),
    (3, "AFTER_COMPLETE_GEOGRAPHY", "retained_after_geography"),
    (4, "AFTER_USABLE_TITLE", "retained_after_title"),
    (5, "AFTER_EXCLUDING_INTERNSHIPS", "retained_after_internship"),
)

stage_array = F.array(
    *[
        F.struct(
            F.lit(stage_order).alias("stage_order"),
            F.lit(stage_name).alias("stage_name"),
            F.col(flag_name).alias("included"),
        )
        for stage_order, stage_name, flag_name in ATTRITION_STAGES
    ]
)

invalid_occupation = (
    F.col("onet_code").isNull()
    | ~F.col("onet_code").isin(*FOCAL_ONET_CODES)
    | ~(
        F.col("onet_code").startswith("17-")
        | F.col("onet_code").startswith("19-")
    )
)

candidate_stage_long = (
    candidate_flagged.select(
        "position_id",
        "user_id",
        "rcid",
        "onet_code",
        F.explode(stage_array).alias("stage"),
    )
    .select("position_id", "user_id", "rcid", "onet_code", "stage.*")
    .filter(F.col("included") == 1)
)

"""
Notes:
(1) explode emits one row for each stage that retains a candidate spell.
(2) One groupBy calculates all requested exact counts for all six stages.
(3) Hidden checks reuse this aggregation to detect duplicate position IDs and bad O*NET codes.
(4) collect is safe here because the result has at most six rows.
"""
attrition_grouped = candidate_stage_long.groupBy("stage_order", "stage_name").agg(
    F.count(F.lit(1)).cast("long").alias("employment_spell_count"),
    F.countDistinct("user_id").cast("long").alias("unique_user_count"),
    F.countDistinct("rcid").cast("long").alias("unique_company_count"),
    F.countDistinct("position_id").cast("long").alias("_distinct_position_id_count"),
    count_true(invalid_occupation, "_invalid_occupation_count"),
)

attrition_rows = attrition_grouped.orderBy("stage_order").collect()
attrition_by_order = {row["stage_order"]: row.asDict() for row in attrition_rows}

print("Sequential Step 3 attrition:")
print("stage | employment spells | unique users | unique companies")
attrition_counts = []
for stage_order, stage_name, _ in ATTRITION_STAGES:
    stage_result = attrition_by_order.get(
        stage_order,
        {
            "employment_spell_count": 0,
            "unique_user_count": 0,
            "unique_company_count": 0,
            "_distinct_position_id_count": 0,
            "_invalid_occupation_count": 0,
        },
    )
    stage_counts = (
        stage_result["employment_spell_count"],
        stage_result["unique_user_count"],
        stage_result["unique_company_count"],
    )
    attrition_counts.append(stage_counts)
    print(
        f"{stage_order} {stage_name} | {stage_counts[0]:,} | "
        f"{stage_counts[1]:,} | {stage_counts[2]:,}"
    )

for earlier_counts, later_counts in zip(attrition_counts, attrition_counts[1:]):
    if any(later > earlier for earlier, later in zip(earlier_counts, later_counts)):
        candidate_flagged.unpersist()
        raise ValueError("Step 3 attrition counts are not weakly decreasing.")

step3_result = attrition_by_order.get(5)
if step3_result is not None:
    if (
        step3_result["employment_spell_count"]
        != step3_result["_distinct_position_id_count"]
    ):
        candidate_flagged.unpersist()
        raise ValueError(
            "Step 3 contains duplicated nonmissing position_id values; "
            "the final tie-break is ambiguous."
        )
    if step3_result["_invalid_occupation_count"] != 0:
        candidate_flagged.unpersist()
        raise ValueError("Step 3 contains a cleaned O*NET code outside the focal universe.")

## Step 4. Retain one focal spell per user-company pair

The preferred grouped `min_by` selects the value struct associated with the smallest complete
ordering struct. A documented window fallback is used only when the installed Spark functions do
not expose `min_by`.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 4. Keep the first observed focal spell for each user at each company
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


SELECTED_VALUE_COLUMNS = (
    "position_id",
    "position_number",
    "start_month",
    "start_year",
    "country",
    "state",
    "title_raw",
    "title_translated",
    "restriction_title",
    "restriction_title_source",
    "seniority",
    "rics_k400",
    "naics_code",
    "naics_description",
    "onet_code",
    "onet_title",
    "reference_onet_title",
)

FINAL_COLUMNS = (
    "position_id",
    "user_id",
    "rcid",
    "position_number",
    "start_month",
    "start_year",
    "country",
    "state",
    "title_raw",
    "title_translated",
    "restriction_title",
    "restriction_title_source",
    "seniority",
    "rics_k400",
    "naics_code",
    "naics_description",
    "onet_code",
    "onet_title",
    "reference_onet_title",
    "n_candidate_focal_spells_user_company",
)

step3_focal_spells = candidate_flagged.filter(
    F.col("retained_after_internship") == 1
).select("user_id", "rcid", *SELECTED_VALUE_COLUMNS)

selection_order = F.struct(
    F.col("start_month").alias("start_month"),
    F.col("seniority").isNull().cast("int").alias("seniority_missing"),
    F.coalesce(F.col("seniority").cast("long"), F.lit(0).cast("long")).alias(
        "seniority_value"
    ),
    F.col("position_number").alias("position_number"),
    F.col("position_id").alias("position_id"),
)

"""
Notes:
(1) The value struct contains only final retained spell fields.
(2) The order is month, observed-before-missing seniority, seniority, position number, and ID.
(3) groupBy performs the necessary shuffle; no extra repartition is added.
(4) The group count records how many Step 3 spells were candidates for selection.
"""
if hasattr(F, "min_by"):
    selection_value = F.struct(
        *[F.col(name).alias(name) for name in SELECTED_VALUE_COLUMNS]
    )
    focal_new_hires_unprojected = (
        step3_focal_spells.groupBy("user_id", "rcid")
        .agg(
            F.count(F.lit(1))
            .cast("long")
            .alias("n_candidate_focal_spells_user_company"),
            F.min_by(selection_value, selection_order).alias("_selected"),
        )
        .select(
            "user_id",
            "rcid",
            "n_candidate_focal_spells_user_company",
            "_selected.*",
        )
    )
    selection_method = "grouped min_by"
else:
    company_window = Window.partitionBy("user_id", "rcid")
    ordered_company_window = company_window.orderBy(
        F.col("start_month").asc(),
        F.col("seniority").asc_nulls_last(),
        F.col("position_number").asc(),
        F.col("position_id").asc(),
    )
    focal_new_hires_unprojected = (
        step3_focal_spells.withColumn(
            "n_candidate_focal_spells_user_company",
            F.count(F.lit(1)).over(company_window),
        )
        .withColumn("_row_number", F.row_number().over(ordered_company_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )
    selection_method = "row_number window fallback"

focal_new_hires = focal_new_hires_unprojected.select(*FINAL_COLUMNS)

missing_final_columns = sorted(set(FINAL_COLUMNS) - set(focal_new_hires.columns))
if missing_final_columns:
    raise ValueError(f"Final output is missing required columns: {missing_final_columns}.")

"""
Notes:
(1) The final projection removes flags, ordering helpers, and raw temporary fields.
(2) Persisting is justified because validation, scientific summaries, and output reuse it.
(3) count materializes the grouped result before the larger candidate cache is released.
"""
focal_new_hires = focal_new_hires.persist(StorageLevel.MEMORY_AND_DISK)
materialized_final_row_count = focal_new_hires.count()
candidate_flagged.unpersist()
print(
    f"Materialized {materialized_final_row_count:,} final rows using {selection_method}; "
    "released the candidate cache."
)

## Final summaries and validations

One aggregate action evaluates the final row-level invariants and requested overall counts. A
second small grouped action reports major-group diagnostics. Failures stop the notebook before
the output path is overwritten.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 5. Validate and summarize the final focal-new-hire sample
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


invalid_start_month = (
    F.col("start_month").isNull()
    | (F.col("start_month") < F.lit(COHORT_START_DATE).cast("date"))
    | (F.col("start_month") >= F.lit(COHORT_END_DATE_EXCLUSIVE).cast("date"))
)
missing_required_key = F.col("user_id").isNull() | F.col("rcid").isNull()
missing_required_identifier = (
    F.col("position_id").isNull()
    | F.col("user_id").isNull()
    | F.col("rcid").isNull()
    | F.col("position_number").isNull()
)
missing_geography_or_title = (
    F.col("country").isNull()
    | F.col("state").isNull()
    | F.col("restriction_title").isNull()
)
internship_in_final = F.coalesce(
    F.col("restriction_title").rlike(INTERNSHIP_TITLE_REGEX),
    F.lit(False),
)
invalid_final_occupation = (
    F.col("onet_code").isNull()
    | ~F.col("onet_code").isin(*FOCAL_ONET_CODES)
    | ~(
        F.col("onet_code").startswith("17-")
        | F.col("onet_code").startswith("19-")
    )
)
invalid_candidate_count = (
    F.col("n_candidate_focal_spells_user_company").isNull()
    | (F.col("n_candidate_focal_spells_user_company") < 1)
)
invalid_start_year = (
    F.col("start_year").isNull()
    | (F.col("start_year") != F.year("start_month"))
)

"""
Notes:
(1) One aggregation combines the requested counts and all row-level invariants.
(2) countDistinct over structs checks composite keys without delimiter-based strings.
(3) Failure counts are exact and return zero even when the final sample is empty.
"""
final_validation = focal_new_hires.agg(
    F.countDistinct("user_id").cast("long").alias("unique_user_count"),
    F.countDistinct("rcid").cast("long").alias("unique_company_count"),
    F.countDistinct(F.struct("user_id", "rcid"))
    .cast("long")
    .alias("distinct_user_company_count"),
    F.countDistinct(F.struct("rcid", "country", "state", "start_year"))
    .cast("long")
    .alias("company_location_cohort_count"),
    F.countDistinct("onet_code").cast("long").alias("distinct_onet_code_count"),
    count_true(missing_required_key, "missing_key_count"),
    count_true(missing_required_identifier, "missing_required_identifier_count"),
    count_true(invalid_start_month, "invalid_start_month_count"),
    count_true(invalid_start_year, "invalid_start_year_count"),
    count_true(missing_geography_or_title, "missing_geography_or_title_count"),
    count_true(internship_in_final, "internship_title_count"),
    count_true(invalid_final_occupation, "invalid_occupation_count"),
    count_true(F.col("reference_onet_title").isNull(), "missing_reference_title_count"),
    count_true(invalid_candidate_count, "invalid_candidate_count"),
).first()

validation_errors = []
if materialized_final_row_count != final_validation["distinct_user_company_count"]:
    validation_errors.append("user_id by rcid does not uniquely identify final rows")
if final_validation["missing_key_count"] != 0:
    validation_errors.append("a final user_id or rcid is missing")
if final_validation["missing_required_identifier_count"] != 0:
    validation_errors.append("a required final identifier is missing")
if final_validation["invalid_start_month_count"] != 0:
    validation_errors.append("a retained start month is outside the cohort range")
if final_validation["invalid_start_year_count"] != 0:
    validation_errors.append("a retained start year is inconsistent with start_month")
if final_validation["missing_geography_or_title_count"] != 0:
    validation_errors.append("a retained country, state, or restriction title is missing")
if final_validation["internship_title_count"] != 0:
    validation_errors.append("a retained title matches the internship expression")
if final_validation["invalid_occupation_count"] != 0:
    validation_errors.append("a retained O*NET code is outside the embedded focal set")
if final_validation["missing_reference_title_count"] != 0:
    validation_errors.append("a retained O*NET code lacks an embedded reference title")
if final_validation["invalid_candidate_count"] != 0:
    validation_errors.append("a user-company candidate count is less than one")

if validation_errors:
    focal_new_hires.unpersist()
    raise ValueError("Final validation failed: " + "; ".join(validation_errors) + ".")

print("Step 4 focal-new-hire summary:")
print(f"Employment spells: {materialized_final_row_count:,}")
print(f"Unique users: {final_validation['unique_user_count']:,}")
print(f"Unique companies: {final_validation['unique_company_count']:,}")
print(
    "Distinct rcid-country-state-start_year cells: "
    f"{final_validation['company_location_cohort_count']:,}"
)

occupation_summary = (
    focal_new_hires.withColumn("onet_major_group", F.substring("onet_code", 1, 2))
    .groupBy("onet_major_group")
    .agg(
        F.count(F.lit(1)).cast("long").alias("employment_spell_count"),
        F.countDistinct("onet_code").cast("long").alias("distinct_onet_code_count"),
    )
    .orderBy("onet_major_group")
    .collect()
)

print("Final O*NET major-group diagnostics:")
for row in occupation_summary:
    print(
        f"Group {row['onet_major_group']}: {row['employment_spell_count']:,} spells, "
        f"{row['distinct_onet_code_count']:,} distinct codes"
    )
print(f"Distinct focal O*NET codes observed: {final_validation['distinct_onet_code_count']:,}")

## Final multi-part Parquet write and cache cleanup

Spark writes the output directory as one distributed Parquet dataset. No managed table,
single-part copy, diagnostics dataset, or hard-coded output partition count is created.

In [ ]:
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>
# <> Step 6. Write the final multi-part Parquet dataset and release the cache
# <>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>#<>


"""
Notes:
(1) overwrite makes reruns reproducible for the same source snapshot and parameters.
(2) The distributed writer preserves the grouped result's default output partitioning.
(3) The finally block releases the final cache even if the storage write fails.
"""
try:
    focal_new_hires.write.mode(OUTPUT_WRITE_MODE).parquet(OUTPUT_PARQUET)
    print(f"Multi-part Parquet dataset saved: {OUTPUT_PARQUET}.")
finally:
    focal_new_hires.unpersist()